# E33 公开眼动数据字段与口径有限核对

2026-10-29。嵌套于E33知识注释；只核行粒度、关键缺失、按脚本字段的完整案例及描述数。不是完整统计复现，不输出个人行，不执行R混合模型。源： https://osf.io/download/g6tym/ ，官方目录 https://api.osf.io/v2/nodes/vzx6q/files/osfstorage/ ，作者脚本 https://osf.io/download/ph4e2/ 。将原名Speed_Dating.sav放在本笔记工作目录；需Python、numpy、pyreadstat（及其依赖）。不把源表纳入公开仓库。

已在本地使用Python 3.12、pyreadstat 1.3.7普通Python进程执行以下代码并捕获真实聚合输出。execution_count留null：未使用Jupyter内核执行或作界面视觉验收。完整案例数不冒充R拟合nobs；Pearson未调整受试/对象聚类，不报p值。质量字段与正文排除口径尚待解释。


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pyreadstat
source = Path("Speed_Dating.sav")
expected = "3de995b8f8abc24908244d770009e600207d6b7940b4a91ff0b82573e45c9f01"
assert hashlib.sha256(source.read_bytes()).hexdigest() == expected
d, meta = pyreadstat.read_sav(source, apply_value_formats=False)
r = {"rows": len(d), "columns": len(d.columns), "subjects": int(d.Subject.nunique()), "partners": int(d.Partner.nunique()), "duplicate_subject_partner": int(d.duplicated(["Subject", "Partner"]).sum())}
r["dates_per_subject"] = d.groupby("Subject").size().value_counts().to_dict()
r["dates_per_partner"] = d.groupby("Partner").size().value_counts().to_dict()
r["choices"] = {"subject_yes": int(d.Subject_Choice.eq(1).sum()), "partner_yes": int(d.Partner_Choice.eq(1).sum()), "matches": int(d.Match.eq(1).sum()), "match_rule_disagreements": int((d.Match.eq(1) != (d.Subject_Choice.eq(1) & d.Partner_Choice.eq(1))).sum())}
cols = ["MutualEyeGaze_duration", "Subject_eyes_duration", "Partner_eyes_duration", "Subject_eyecontact", "Partner_Attractiveness", "Subject_Attractiveness", "Attractiveness_mean"]
r["missing"] = {c: int(d[c].isna().sum()) for c in cols}
models = {"m1": ["Subject_Choice", "Gender", "Subject", "MutualEyeGaze_duration"], "m3": ["Subject_Choice", "Gender", "Subject", "Subject_eyecontact"], "m4": ["Subject_Choice", "Gender", "Subject", "MutualEyeGaze_duration", "Attractiveness_mean"], "m6": ["Subject_Choice", "Gender", "Subject", "Partner_eyes_duration", "Partner_Attractiveness"]}
r["complete_cases_by_script_fields_not_fitted_nobs"] = {k: int(d[v].notna().all(axis=1).sum()) for k,v in models.items()}
# Convert numeric SAV categories using the source value labels, not by guessing scale.
labels = meta.variable_value_labels["Subject_eyecontact"]
x = d.Subject_eyecontact.map({k: float(v.rstrip("%")) for k,v in labels.items()})
r["subjective_percent_mean"] = float(x.mean())
r["mutual_duration_mean_seconds"] = float(d.MutualEyeGaze_duration.mean())
r["unclustered_pearson_not_pvalues"] = {c: float(x.corr(d[c])) for c in ["MutualEyeGaze_duration", "Subject_eyes_duration", "Partner_eyes_duration"]}
# These fields have 'Data Quality' labels; their connection to the exclusion rule is unresolved.
r["below80_by_subject_quality_field"] = int(d.Subject_GazeSample.lt(80).sum())
r["below80_by_partner_quality_field"] = int(d.Partner_GazeSample.lt(80).sum())
r["below80_either_quality_field"] = int((d.Subject_GazeSample.lt(80) | d.Partner_GazeSample.lt(80)).sum())
r["eye_missing_rows_all_below80_either"] = bool((d.Subject_GazeSample.lt(80) | d.Partner_GazeSample.lt(80))[d.MutualEyeGaze_duration.isna()].all())
r["attract_mean_matches_available_component_mean"] = bool(np.allclose(d.Attractiveness_mean, d[["Subject_Attractiveness", "Partner_Attractiveness"]].mean(axis=1), equal_nan=True))
r["one_component_missing_rows"] = int(d[["Subject_Attractiveness", "Partner_Attractiveness"]].isna().any(axis=1).sum())
print(json.dumps(r, ensure_ascii=False, indent=2))


{
  "rows": 240,
  "columns": 31,
  "subjects": 60,
  "partners": 8,
  "duplicate_subject_partner": 0,
  "dates_per_subject": {
    "4": 60
  },
  "dates_per_partner": {
    "30": 8
  },
  "choices": {
    "subject_yes": 72,
    "partner_yes": 43,
    "matches": 10,
    "match_rule_disagreements": 0
  },
  "missing": {
    "MutualEyeGaze_duration": 3,
    "Subject_eyes_duration": 3,
    "Partner_eyes_duration": 3,
    "Subject_eyecontact": 1,
    "Partner_Attractiveness": 12,
    "Subject_Attractiveness": 8,
    "Attractiveness_mean": 0
  },
  "complete_cases_by_script_fields_not_fitted_nobs": {
    "m1": 237,
    "m3": 239,
    "m4": 237,
    "m6": 225
  },
  "subjective_percent_mean": 64.81171548117155,
  "mutual_duration_mean_seconds": 24.76185654008439,
  "unclustered_pearson_not_pvalues": {
    "MutualEyeGaze_duration": 0.15794329217716632,
    "Subject_eyes_duration": 0.14152752357928222,
    "Partner_eyes_duration": 0.09238913132583094
  },
  "below80_by_subject_quality_field": 